# 🎙️ VoiceBatch Studio Pro (Clone Edition)
अब आप अपनी या किसी की भी आवाज़ क्लोन कर सकते हैं। 100% फ्री और रियलिस्टिक।

In [ ]:
# @title 📥 Step 1: हाई-प्रोफाइल वॉइस इंजन इंस्टॉल करें
print("⏳ XTTS v2 और क्लोनिंग इंजन लोड हो रहा है (इसमें 2-3 मिनट लग सकते हैं)...")
!pip install -q gradio edge-tts TTS
print("✅ इंजन तैयार है!")

In [ ]:
# @title 🚀 Step 2: प्रो स्टूडियो लॉन्च करें
import gradio as gr
import torch
from TTS.api import TTS
import os
import asyncio
import edge_tts

# डिवाइस चेक (GPU है तो बहुत तेज़ चलेगा)
device = "cuda" if torch.cuda.is_available() else "cpu"

# क्लोनिंग मॉडल लोड करना
print("📥 मॉडल लोड हो रहा है...")
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

def clone_voice(text, audio_file):
    output_path = "cloned_output.wav"
    tts_model.tts_to_file(text=text, speaker_wav=audio_file, language="hi", file_path=output_path)
    return output_path

async def fast_tts(text, voice):
    output = "edge_output.mp3"
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(output)
    return output

with gr.Blocks(theme=gr.themes.Default(primary_hue="orange")) as demo:
    gr.Markdown("# 🎙️ VoiceBatch Studio Pro")
    
    with gr.Tabs():
        with gr.TabItem("👤 Voice Cloning (आवाज़ की नकल)"):
            gr.Markdown("### किसी भी आवाज़ का 6 सेकंड का सैंपल डालें और उसे क्लोन करें।")
            with gr.Row():
                with gr.Column():
                    clone_text = gr.Textbox(label="क्या कहलवाना है?", lines=4)
                    sample_audio = gr.Audio(label="जिसकी आवाज़ चाहिए उसका सैंपल डालें", type="filepath")
                    clone_btn = gr.Button("Clone & Generate 🚀", variant="primary")
                output_clone = gr.Audio(label="क्लोन की हुई आवाज़")
            
            clone_btn.click(clone_voice, [clone_text, sample_audio], output_clone)

        with gr.TabItem("⚡ Fast TTS (Standard)"):
            gr.Markdown("### हाई-स्पीड रियलिस्टिक आवाज़ें।")
            with gr.Row():
                with gr.Column():
                    std_text = gr.Textbox(label="टेक्स्ट लिखें", lines=4)
                    v_choice = gr.Dropdown(label="आवाज़ चुनें", choices=["hi-IN-MadhurNeural", "hi-IN-SwaraNeural", "en-US-GuyNeural"])
                    std_btn = gr.Button("आवाज़ बनाएँ")
                output_std = gr.Audio(label="ऑडियो")
            
            std_btn.click(lambda t, v: asyncio.run(fast_tts(t, v)), [std_text, v_choice], output_std)

demo.launch(share=True)